### count virus and cell

In [ ]:
import pandas as pd



input_file = r"Y:\Huihui\Mouse_Analysis\04022026\Mouse1\Brain2_1\STARmap_output\Barcodelist.csv"
output_file = r"C:\Users\Manjari\Downloads\virus_gene_counts_summary2.xlsx"

df = pd.read_csv(input_file)


df.columns = df.columns.str.strip()


cell_id_col = "cell_globID"
virus_col = "virus"
gene_col = "Names"


df_gene = df.dropna(subset=[gene_col]).copy()

df_gene = df_gene[df_gene[gene_col].astype(str).str.strip() != ""]



virus_cell_counts = (
    df.groupby(virus_col)[cell_id_col]
    .nunique()
    .reset_index(name="num_cells")
)



gene_counts_long = (
    df_gene.groupby([cell_id_col, virus_col, gene_col])
    .size()
    .reset_index(name="gene_count")
)



gene_counts_wide = (
    gene_counts_long
    .pivot_table(
        index=[cell_id_col, virus_col],
        columns=gene_col,
        values="gene_count",
        fill_value=0
    )
    .reset_index()
)

gene_counts_wide.columns.name = None



all_cells = df[[cell_id_col, virus_col]].drop_duplicates()

gene_counts_wide = all_cells.merge(
    gene_counts_wide,
    on=[cell_id_col, virus_col],
    how="left"
)


gene_cols = [
    c for c in gene_counts_wide.columns
    if c not in [cell_id_col, virus_col]
]


gene_counts_wide[gene_cols] = (
    gene_counts_wide[gene_cols]
    .fillna(0)
    .astype(int)
)



with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    virus_cell_counts.to_excel(
        writer,
        sheet_name="virus_cell_counts",
        index=False
    )

    gene_counts_long.to_excel(
        writer,
        sheet_name="gene_counts_long",
        index=False
    )

    gene_counts_wide.to_excel(
        writer,
        sheet_name="gene_counts_wide",
        index=False
    )

print(f"Saved to: {output_file}")

Saved to: C:\Users\Manjari\Downloads\virus_gene_counts_summary2.xlsx


In [ ]:
import pandas as pd
import numpy as np



cell_id_col = "cell_globID"
virus_col = "virus"

marker_genes = ["GFAP", "Gad1", "TMEM119", "Vglut1"]

min_counts = 3


for gene in marker_genes:
    if gene not in gene_counts_wide.columns:
        gene_counts_wide[gene] = 0

gene_counts_wide[marker_genes] = (
    gene_counts_wide[marker_genes]
    .fillna(0)
    .astype(int)
)



def assign_cell_type(row, marker_genes, min_counts=3):
    counts = row[marker_genes]

    max_count = counts.max()

    if max_count < min_counts:
        return "NA"


    top_genes = counts[counts == max_count].index.tolist()

    if len(top_genes) > 1:
        return "unknown"

    return top_genes[0]


gene_counts_wide["cell_type"] = gene_counts_wide.apply(
    assign_cell_type,
    axis=1,
    marker_genes=marker_genes,
    min_counts=min_counts
)


virus_celltype_counts = (
    gene_counts_wide
    .groupby([virus_col, "cell_type"])[cell_id_col]
    .nunique()
    .reset_index(name="num_cells")
)

print(virus_celltype_counts)

virus_celltype_counts_wide = (
    virus_celltype_counts
    .pivot_table(
        index=virus_col,
        columns="cell_type",
        values="num_cells",
        fill_value=0
    )
    .reset_index()
)

virus_celltype_counts_wide.columns.name = None

print(virus_celltype_counts_wide)


output_file = r"C:\Users\Manjari\Downloads\virus_celltype_summary2.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    gene_counts_wide.to_excel(
        writer,
        sheet_name="cell_type_per_cell",
        index=False
    )

    virus_celltype_counts.to_excel(
        writer,
        sheet_name="virus_celltype_long",
        index=False
    )

    virus_celltype_counts_wide.to_excel(
        writer,
        sheet_name="virus_celltype_wide",
        index=False
    )

print(f"Saved to: {output_file}")

       virus cell_type  num_cells
0       EEEV      GFAP          7
1       EEEV      Gad1          6
2       EEEV        NA         22
3       EEEV    Vglut1         56
4       EEEV   unknown          1
5        SFV      Gad1          3
6        SFV        NA          3
7        SFV    Vglut1          2
8        SFV   unknown          1
9       SINV      GFAP         12
10      SINV      Gad1         31
11      SINV        NA         21
12      SINV    Vglut1        190
13      SINV   unknown          8
14      VEEV      GFAP         78
15      VEEV      Gad1         33
16      VEEV        NA        125
17      VEEV   TMEM119          1
18      VEEV    Vglut1        190
19      VEEV   unknown         20
20      WEEV        NA          1
21      WEEV    Vglut1          1
22      WEEV   unknown          1
23  no_gene4      GFAP          3
24  no_gene4    Vglut1          1
      virus  GFAP  Gad1     NA  TMEM119  Vglut1  unknown
0      EEEV   7.0   6.0   22.0      0.0    56.0      1.0
1 

In [ ]:
import pandas as pd
import numpy as np


input_file = r"Y:\Huihui\Mouse_Analysis\04022026\Mouse4\virus_celltype_summary2.xlsx"
output_file = r"Y:\Huihui\Mouse_Analysis\04022026\Mouse4\virus_celltype_summary2_06162026.xlsx"

cell_id_col = "cell_globID"
virus_col = "virus"

marker_genes = ["GFAP", "Gad1", "TMEM119", "Vglut1"]

min_counts = 3
ratio_cutoff = 0.67


gene_counts_wide = pd.read_excel(input_file)

gene_counts_wide.columns = gene_counts_wide.columns.str.strip()


for gene in marker_genes:
    if gene not in gene_counts_wide.columns:
        gene_counts_wide[gene] = 0

gene_counts_wide[marker_genes] = (
    gene_counts_wide[marker_genes]
    .fillna(0)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)


def assign_cell_type(row, marker_genes, min_counts=3, ratio_cutoff=0.67):
    counts = row[marker_genes]

    max_count = counts.max()

    if max_count < min_counts:
        return "NA"

    top_genes = counts[counts == max_count].index.tolist()

    if len(top_genes) > 1:
        return "unknown"

    top_gene = top_genes[0]

    gad1 = row["Gad1"]
    vglut1 = row["Vglut1"]

    if top_gene == "Vglut1":
        ratio = gad1 / vglut1

        if ratio < ratio_cutoff:
            return "excitatory neuron"
        else:
            return "unknown"

    elif top_gene == "Gad1":
        ratio = vglut1 / gad1

        if ratio < ratio_cutoff:
            return "inhibitory neuron"
        else:
            return "unknown"

    else:
        return top_gene


gene_counts_wide["cell_type"] = gene_counts_wide.apply(
    assign_cell_type,
    axis=1,
    marker_genes=marker_genes,
    min_counts=min_counts,
    ratio_cutoff=ratio_cutoff
)


virus_celltype_counts = (
    gene_counts_wide
    .groupby([virus_col, "cell_type"])[cell_id_col]
    .nunique()
    .reset_index(name="num_cells")
)

print(virus_celltype_counts)


virus_celltype_counts_wide = (
    virus_celltype_counts
    .pivot_table(
        index=virus_col,
        columns="cell_type",
        values="num_cells",
        fill_value=0
    )
    .reset_index()
)

virus_celltype_counts_wide.columns.name = None

print(virus_celltype_counts_wide)


with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    gene_counts_wide.to_excel(
        writer,
        sheet_name="cell_type_per_cell",
        index=False
    )

    virus_celltype_counts.to_excel(
        writer,
        sheet_name="virus_celltype_long",
        index=False
    )

    virus_celltype_counts_wide.to_excel(
        writer,
        sheet_name="virus_celltype_wide",
        index=False
    )

print(f"Saved to: {output_file}")

       virus          cell_type  num_cells
0       EEEV                 NA          3
1       EEEV  excitatory neuron        157
2       EEEV  inhibitory neuron          9
3       EEEV            unknown          6
4        SFV               GFAP          1
5        SFV                 NA          1
6        SFV  excitatory neuron          7
7       SINV                 NA          3
8       SINV  excitatory neuron        166
9       SINV  inhibitory neuron          6
10      SINV            unknown          3
11      VEEV               GFAP         31
12      VEEV                 NA         99
13      VEEV  excitatory neuron        195
14      VEEV  inhibitory neuron         24
15      VEEV            unknown         17
16      WEEV                 NA          0
17      WEEV  excitatory neuron          8
18      WEEV  inhibitory neuron          1
19      WEEV            unknown          1
20  no_gene4                 NA          2
21  no_gene4  excitatory neuron          5
      virus